In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PROJECT_ROOT = Path(r"C:\Users\asus\OneDrive\EV-projects\evcs-projects")
BENCH_DIR    = PROJECT_ROOT / "results" / "benchmarking"
EXCEL_FILE   = BENCH_DIR / "benchmark_with_SLURM.xlsx"

df = pd.read_excel(EXCEL_FILE, sheet_name="benchmark")
print(f"Loaded {len(df)} rows")
df[['Timestamp','Instance','N','T','D_km','seed','DR_best','Exact_incumbent_raw','Gap_%']]

Loaded 10 rows


,Timestamp,Instance,N,T,D_km,seed,DR_best,Exact_incumbent_raw,Gap_%
0,2026-04-03 16:47:58,center_146_Verona_k250,250,3,0.5,11,385.2353,402.725042,4.2865
1,2026-04-03 17:00:01,center_146_Verona_k250,250,3,0.5,11,385.3952,402.725042,4.2467
2,2026-04-03 17:13:07,center_146_Verona_k250,250,3,0.5,11,385.3952,402.725042,4.2467
3,2026-04-03 18:21:30,center_146_Verona_k250,250,6,2.0,11,1408.6489,1410.361659,-0.1988
4,2026-04-03 20:06:46,center_146_Verona_k250,250,6,0.5,11,984.4594,1084.305270,9.1486
5,2026-04-03 20:12:38,center_146_Verona_k250,250,6,0.5,11,981.9694,1084.305270,9.3784
6,2026-04-03 21:15:49,center_146_Verona_k250,250,6,2.0,11,1409.9914,1410.437016,-0.2943
7,2026-04-03 22:38:59,center_146_Verona_k250,250,6,2.0,11,1409.9914,1410.437016,-0.2943
8,2026-04-04 00:13:40,center_146_Verona_k250,250,6,2.0,11,1408.6489,1410.290162,-0.1988
9,2026-04-04 15:06:04,center_146_Verona_k250,250,6,2.0,11,1407.9797,1410.290162,0.1638


In [ ]:
# --- build a short label for each row ---
def row_label(row, idx):
    return f"#{idx}  T={int(row['T'])} D={row['D_km']} seed={int(row['seed'])}"

df["label"] = [row_label(r, i) for i, r in df.iterrows()]

print("Labels ready.")
df[["label", "DR_best", "Exact_incumbent_raw", "Gap_%"]]


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
labels = df["label"].tolist()
x = np.arange(len(df))
width = 0.35

# ── Panel 1: DR_best vs Exact_incumbent_raw ──────────────────────────────────
ax1 = axes[0]
bars_dr    = ax1.bar(x - width/2, df["DR_best"],              width, label="DR best",       color="tab:orange", alpha=0.85)
bars_exact = ax1.bar(x + width/2, df["Exact_incumbent_raw"],  width, label="Exact incumbent", color="tab:blue",   alpha=0.85)

# value labels on bars
for bar in bars_dr:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}",
             ha="center", va="bottom", fontsize=7, color="tab:orange")
for bar in bars_exact:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}",
             ha="center", va="bottom", fontsize=7, color="tab:blue")

ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax1.set_ylabel("Objective value (covered demand)", fontsize=9)
ax1.set_title("DR best vs Exact incumbent — all benchmark rows", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(axis="y", linewidth=0.5, alpha=0.6)

# ── Panel 2: Gap_% bar chart ─────────────────────────────────────────────────
ax2 = axes[1]
gaps   = df["Gap_%"].tolist()
colors = ["tab:green" if g <= 1.0 else "tab:orange" if g <= 5.0 else "tab:red"
          for g in gaps]
bars_gap = ax2.bar(x, gaps, color=colors, alpha=0.85, edgecolor="white")

# value labels
for bar, g in zip(bars_gap, gaps):
    va  = "bottom" if g >= 0 else "top"
    off = 0.05 if g >= 0 else -0.05
    ax2.text(bar.get_x() + bar.get_width()/2, g + off, f"{g:.2f}%",
             ha="center", va=va, fontsize=8)

ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax2.set_ylabel("Gap  (Exact − DR) / Exact  [%]", fontsize=9)
ax2.set_title("Optimality gap per run   (green ≤ 1%  |  orange ≤ 5%  |  red > 5%)", fontsize=11)
ax2.grid(axis="y", linewidth=0.5, alpha=0.6)

# legend patches
from matplotlib.patches import Patch
ax2.legend(handles=[
    Patch(color="tab:green",  label="gap ≤ 1%"),
    Patch(color="tab:orange", label="1% < gap ≤ 5%"),
    Patch(color="tab:red",    label="gap > 5%"),
], fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# --- summary table with color-coded gap ---
cols = ['Timestamp','Instance','Policy','N','T','D_km','seed',
        'Exact_incumbent_raw','DR_best','Gap_%','DR_iters','DR_time_s','Exact_time_s']
show = [c for c in cols if c in df.columns]
df[show].style \
    .format({'Exact_incumbent_raw':'{:.4f}','DR_best':'{:.4f}',
             'Gap_%':'{:.4f}%','DR_time_s':'{:.1f}s','Exact_time_s':'{:.1f}s'}) \
    .background_gradient(subset=['Gap_%'], cmap='RdYlGn_r')


In [5]:
# --- summary table with color-coded gap ---
cols = ['Timestamp','Instance','Policy','N','T','D_km','seed',
        'Exact_incumbent_raw','DR_best','Gap_%','DR_iters','DR_time_s','Exact_time_s']
show = [c for c in cols if c in df.columns]
df[show].style \
    .format({'Exact_incumbent_raw':'{:.4f}','DR_best':'{:.4f}',
             'Gap_%':'{:.4f}%','DR_time_s':'{:.1f}s','Exact_time_s':'{:.1f}s'}) \
    .background_gradient(subset=['Gap_%'], cmap='RdYlGn_r')

,Timestamp,Instance,Policy,N,T,D_km,seed,Exact_incumbent_raw,DR_best,Gap_%,DR_iters,DR_time_s,Exact_time_s
0,2026-04-03 16:47:58,center_146_Verona_k250,closest_priority,250,3,0.500000,11,402.7250,385.2353,4.2865%,545,601.5s,0.8s
1,2026-04-03 17:00:01,center_146_Verona_k250,closest_priority,250,3,0.500000,11,402.7250,385.3952,4.2467%,558,600.3s,0.5s
2,2026-04-03 17:13:07,center_146_Verona_k250,closest_priority,250,3,0.500000,11,402.7250,385.3952,4.2467%,564,601.7s,0.6s
3,2026-04-03 18:21:30,center_146_Verona_k250,closest_priority,250,6,2.000000,11,1410.3617,1408.6489,-0.1988%,33,619.6s,305.1s
4,2026-04-03 20:06:46,center_146_Verona_k250,closest_priority,250,6,0.500000,11,1084.3053,984.4594,9.1486%,189,602.0s,10.6s
5,2026-04-03 20:12:38,center_146_Verona_k250,closest_priority,250,6,0.500000,11,1084.3053,981.9694,9.3784%,174,602.2s,7.1s
6,2026-04-03 21:15:49,center_146_Verona_k250,closest_priority,250,6,2.000000,11,1410.4370,1409.9914,-0.2943%,121,3005.4s,459.1s
7,2026-04-03 22:38:59,center_146_Verona_k250,closest_priority,250,6,2.000000,11,1410.4370,1409.9914,-0.2943%,134,3006.0s,341.9s
8,2026-04-04 00:13:40,center_146_Verona_k250,closest_priority,250,6,2.000000,11,1410.2902,1408.6489,-0.1988%,46,3013.5s,618.9s
9,2026-04-04 15:06:04,center_146_Verona_k250,closest_priority,250,6,2.000000,11,1410.2902,1407.9797,0.1638%,14,51665.5s,616.9s
